# Cargamos librerías

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Cargamos y limpiamos DF

In [11]:
# Cargar el archivo CSV en un DataFrame
df = pd.read_csv('datos_los_train.csv', index_col='eid').drop(columns=['vdate','facid'])
    
# Mapear género (F-0, M-1)
df['gender'] = df['gender'].map({'F': 0, 'M': 1})

# Arreglar readmission count, si es 5+ lo establecemos como 6
df['rcount'] = df['rcount'].replace('5+', '6').astype(int)

# Dividimos en X e y train y test

In [12]:
X = df.drop(columns=['lengthofstay'])
y = df['lengthofstay']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


print(f"Filas para Train: {X_train.shape[0]}")
print(f"Filas para Test: {X_test.shape[0]}")

Filas para Train: 64053
Filas para Test: 16014


In [3]:
X_train.columns

Index(['rcount', 'gender', 'dialysisrenalendstage', 'asthma', 'irondef',
       'pneum', 'substancedependence', 'psychologicaldisordermajor', 'depress',
       'psychother', 'fibrosisandother', 'malnutrition', 'hemo', 'hematocrit',
       'neutrophils', 'sodium', 'glucose', 'bloodureanitro', 'creatinine',
       'bmi', 'pulse', 'respiration', 'secondarydiagnosisnonicd9'],
      dtype='object')

# Construimos y entrenamos el modelo

In [ ]:
# Construimos modelo
modelo = CatBoostRegressor(
    iterations=3000,          
    learning_rate=0.05,       # Un poco más rápido para no eternizarnos
    random_seed=42,
    od_type='Iter',           # Detector de Overfitting
    od_wait=100,              # Si pasan 100 árboles sin mejorar el test, frena
    verbose=250               # Imprime por pantalla cada 250 iteraciones
)

modelo.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    use_best_model=True       # Automáticamente se queda con la iteración que dio el MAE más bajo
)

# Hacer predicciones sobre los datos de validación
print("Hacemos predicciones")
predicciones = modelo.predict(X_test)

# Calcular los errores
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))

print("RESULTADOS:")
print(f"MAE: {mae:.2f} días")
print(f"RMSE: {rmse:.2f} días")


0:	learn: 2.2758211	test: 2.2781084	best: 2.2781084 (0)	total: 149ms	remaining: 7m 25s
250:	learn: 0.4616659	test: 0.4702898	best: 0.4702898 (250)	total: 2.15s	remaining: 23.6s
500:	learn: 0.3982679	test: 0.4132237	best: 0.4132237 (500)	total: 4.64s	remaining: 23.1s
750:	learn: 0.3806152	test: 0.4011037	best: 0.4010985 (749)	total: 6.56s	remaining: 19.7s
1000:	learn: 0.3704676	test: 0.3965431	best: 0.3965431 (1000)	total: 8.55s	remaining: 17.1s
1250:	learn: 0.3628879	test: 0.3944614	best: 0.3944601 (1247)	total: 10.5s	remaining: 14.6s
1500:	learn: 0.3564546	test: 0.3936025	best: 0.3935997 (1499)	total: 12.5s	remaining: 12.5s
1750:	learn: 0.3505410	test: 0.3928342	best: 0.3928118 (1735)	total: 14.5s	remaining: 10.3s
2000:	learn: 0.3454409	test: 0.3926916	best: 0.3926208 (1950)	total: 16.5s	remaining: 8.26s
2250:	learn: 0.3407011	test: 0.3922982	best: 0.3922866 (2244)	total: 18.5s	remaining: 6.15s
2500:	learn: 0.3360295	test: 0.3921166	best: 0.3921166 (2500)	total: 20.4s	remaining: 4.07s

In [9]:
print(f"MAE en horas: {mae*24:.2f}h")

MAE en horas: 7.03h


In [13]:
import pandas as pd
import pprint # Para imprimir diccionarios de forma bonita

print("--- 1. ANATOMÍA DEL MODELO ---")
# ¿Cuántos árboles construyó al final?
print(f"Árboles construidos: {modelo.tree_count_}")
# ¿En qué iteración exacta consiguió el mejor resultado antes de que saltara el Early Stopping?
print(f"Mejor iteración: {modelo.get_best_iteration()}")

print("\n--- 2. LOS PARÁMETROS OCULTOS ---")
# Esto te devuelve TODO lo que CatBoost configuró por debajo (incluso lo que no le dijiste)
parametros_internos = modelo.get_all_params()
print("Algunos parámetros clave que usó el motor:")
print(f"- Profundidad de los árboles (depth): {parametros_internos.get('depth')}")
print(f"- Penalización L2 (l2_leaf_reg): {parametros_internos.get('l2_leaf_reg')}")
print(f"- Tipo de crecimiento: {parametros_internos.get('grow_policy')}")

print("\n--- 3. IMPORTANCIA DE LAS VARIABLES (El chivato) ---")
# Esto te dice qué peso % tiene cada columna para decidir los días de ingreso
importancias = modelo.get_feature_importance()

# Lo metemos en una tabla para que sea fácil de leer
df_importancias = pd.DataFrame({
    'Variable': X_train.columns,
    'Peso_Porcentual': importancias
})

# Lo ordenamos de mayor a menor
df_importancias = df_importancias.sort_values(by='Peso_Porcentual', ascending=False)
print(df_importancias.head(10)) # Mostramos el Top 10

--- 1. ANATOMÍA DEL MODELO ---
Árboles construidos: 2617
Mejor iteración: 2616

--- 2. LOS PARÁMETROS OCULTOS ---
Algunos parámetros clave que usó el motor:
- Profundidad de los árboles (depth): 6
- Penalización L2 (l2_leaf_reg): 3
- Tipo de crecimiento: SymmetricTree

--- 3. IMPORTANCIA DE LAS VARIABLES (El chivato) ---
                      Variable  Peso_Porcentual
0                       rcount        35.462572
7   psychologicaldisordermajor        11.946839
13                  hematocrit         6.232911
21                 respiration         5.496828
19                         bmi         4.142756
15                      sodium         4.132414
16                     glucose         4.108118
18                  creatinine         4.043512
20                       pulse         4.023743
12                        hemo         3.501153


In [5]:
# Extraemos los valores y calculamos la media como un número simple
media_real = float(y_test.values.mean())

print(f"Estancia media real de los pacientes: {media_real:.2f} días")
print(f"Error porcentual aproximado: {(mae / media_real) * 100:.2f}%")

Estancia media real de los pacientes: 4.00 días
Error porcentual aproximado: 7.32%
